In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 150

# ====================================
# Load data and focus on modern era
# ====================================
df = pd.read_csv("nhl_games_with_nst_all_odds_rolling_features.csv")

df["game_date"] = pd.to_datetime(df["game_date"])
# keep games where Pinnacle odds exist (roughly 2020+)
eda_df = df[df["pinnacle_ml_prob_home_fair"].notna()].copy()
print("EDA sample size:", eda_df.shape)

# ====================================
# TABLE 1: Summary stats for key vars
# ====================================

key_numeric = [
    "home_goals",
    "away_goals",
    "home_win_margin",
    "pinnacle_ml_prob_home_fair",
    "pinnacle_total_line",
    "diff_xGFPct_roll10",
    "diff_CFPct_roll10",
    "diff_HDCFPct_roll10",
    "diff_win_pct_roll10",
    "diff_goal_diff_roll10",
]

# keep only those that actually exist
key_numeric = [c for c in key_numeric if c in eda_df.columns]
print("Using for Table 1:", key_numeric)

t1_df = eda_df[key_numeric].dropna().copy()
summary = t1_df.agg(["mean", "median", "std", "min", "max"]).T
summary["skewness"] = t1_df.apply(lambda x: skew(x, nan_policy="omit"))
summary["kurtosis"] = t1_df.apply(lambda x: kurtosis(x, nan_policy="omit"))

print("\nTABLE 1: Summary statistics")
print(summary)

summary.to_csv("table1_summary_stats.csv")

# ====================================
# TABLE 2: Correlation matrix
# ====================================

corr_vars = [
    "home_win",                         # outcome
    "home_goals",
    "away_goals",
    "home_win_margin",
    "pinnacle_ml_prob_home_fair",
    "pinnacle_total_line",
    "diff_xGFPct_roll10",
    "diff_CFPct_roll10",
    "diff_HDCFPct_roll10",
    "diff_win_pct_roll10",
    "diff_goal_diff_roll10",
]

corr_vars = [c for c in corr_vars if c in eda_df.columns]
corr_df = eda_df[corr_vars].dropna().copy()

corr_matrix = corr_df.corr()
print("\nTABLE 2: Correlation matrix")
print(corr_matrix)

corr_matrix.to_csv("table2_corr_matrix.csv")

# ====================================
# FIGURE 1: Home vs Away win counts
# ====================================

fig, ax = plt.subplots(figsize=(7, 5))
win_counts = eda_df["home_win"].value_counts().sort_index()
# index: 0 = away win, 1 = home win
ax.bar(["Away win (0)", "Home win (1)"], win_counts.values)
ax.set_ylabel("Number of games")
ax.set_title("Distribution of Home vs Away Wins (2020–2024)")
for i, v in enumerate(win_counts.values):
    ax.text(i, v + 50, str(v), ha="center", va="bottom", fontsize=10)
fig.tight_layout()
fig.savefig("figure1_home_vs_away_wins.png")
plt.close(fig)

# ====================================
# FIGURE 2: Total goals distribution
# ====================================

if {"home_goals", "away_goals"} <= set(eda_df.columns):
    eda_df["total_goals"] = eda_df["home_goals"] + eda_df["away_goals"]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(eda_df["total_goals"].dropna(), bins=range(0, 15))
    ax.set_xlabel("Total goals in game")
    ax.set_ylabel("Frequency")
    ax.set_title("Distribution of Total Goals per Game (2020–2024)")
    fig.tight_layout()
    fig.savefig("figure2_total_goals_hist.png")
    plt.close(fig)

# ====================================
# FIGURE 3: Pinnacle fair prob by outcome
# ====================================

fig, ax = plt.subplots(figsize=(7, 5))
plot_df = eda_df[["home_win", "pinnacle_ml_prob_home_fair"]].dropna().copy()
plot_df["home_win"] = plot_df["home_win"].astype(int)

sns.boxplot(
    x="home_win",
    y="pinnacle_ml_prob_home_fair",
    data=plot_df,
    ax=ax
)
ax.set_xticklabels(["Away win (0)", "Home win (1)"])
ax.set_xlabel("Game outcome")
ax.set_ylabel("Pinnacle fair implied home-win probability")
ax.set_title("Pinnacle Fair Home-Win Probability by Outcome")
fig.tight_layout()
fig.savefig("figure3_pinnacle_prob_by_outcome.png")
plt.close(fig)

# ====================================
# FIGURE 4: Scatter – form vs odds
# ====================================

if {"diff_xGFPct_roll10", "pinnacle_ml_prob_home_fair"} <= set(eda_df.columns):
    fig, ax = plt.subplots(figsize=(7, 5))
    scatter_df = eda_df[["diff_xGFPct_roll10", "pinnacle_ml_prob_home_fair"]].dropna()
    ax.scatter(
        scatter_df["diff_xGFPct_roll10"],
        scatter_df["pinnacle_ml_prob_home_fair"],
        alpha=0.4
    )
    ax.set_xlabel("10-game xG% differential (home − away)")
    ax.set_ylabel("Pinnacle fair implied home-win probability")
    ax.set_title("Recent xG Form vs Pinnacle Fair Home-Win Probability")
    fig.tight_layout()
    fig.savefig("figure4_xg_diff_vs_pinnacle_prob.png")
    plt.close(fig)

# ====================================
# FIGURE 5: Correlation heatmap
# ====================================

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=False,
    cmap="coolwarm",
    center=0,
    ax=ax
)
ax.set_title("Correlation Heatmap for Outcome, Odds, and Form Metrics")
fig.tight_layout()
fig.savefig("figure5_corr_heatmap.png")
plt.close(fig)

print("\nSaved:")
print("- table1_summary_stats.csv")
print("- table2_corr_matrix.csv")
print("- figure1_home_vs_away_wins.png")
print("- figure2_total_goals_hist.png")
print("- figure3_pinnacle_prob_by_outcome.png")
print("- figure4_xg_diff_vs_pinnacle_prob.png")
print("- figure5_corr_heatmap.png")


EDA sample size: (4770, 513)
Using for Table 1: ['home_goals', 'away_goals', 'home_win_margin', 'pinnacle_ml_prob_home_fair', 'pinnacle_total_line', 'diff_xGFPct_roll10', 'diff_CFPct_roll10', 'diff_HDCFPct_roll10', 'diff_win_pct_roll10', 'diff_goal_diff_roll10']

TABLE 1: Summary statistics
                                mean    median        std         min  \
home_goals                  3.176138  3.000000   1.775849    0.000000   
away_goals                  2.953350  3.000000   1.686326    0.000000   
home_win_margin             0.222788  1.000000   2.594288   -7.000000   
pinnacle_ml_prob_home_fair  0.532533  0.535484   0.134012    0.052757   
pinnacle_total_line         6.008058  6.000000   0.659944    2.500000   
diff_xGFPct_roll10          0.017821  0.020000   5.588054  -17.190000   
diff_CFPct_roll10           0.041992  0.030000   4.883683  -16.810000   
diff_HDCFPct_roll10        -0.004050 -0.014444   6.129625  -17.490000   
diff_win_pct_roll10         0.008199  0.000000   0.

/var/folders/fz/0282wvb93rn0lm_p0nscw1sm0000gn/T/ipykernel_23155/4158749430.py:123: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(["Away win (0)", "Home win (1)"])



Saved:
- table1_summary_stats.csv
- table2_corr_matrix.csv
- figure1_home_vs_away_wins.png
- figure2_total_goals_hist.png
- figure3_pinnacle_prob_by_outcome.png
- figure4_xg_diff_vs_pinnacle_prob.png
- figure5_corr_heatmap.png
